In [3]:
# %%
# --- setup
from pathlib import Path
import pandas as pd
from collections import Counter
import re, hashlib, json, math, random

# repo-aware paths
ROOT = Path.cwd().parent
DATA = (ROOT / "data") if (ROOT / "data").exists() else ROOT  # be forgiving
print(f"Using data path: {DATA}")
FILES = {
    "clean_parquet": DATA / "ashaar.clean.parquet",
    "split_train":   DATA / "ashaar.train.parquet",
    "split_dev":     DATA / "ashaar.dev.parquet",
    "split_test":    DATA / "ashaar.test.parquet",
    "tapt_train_diac":   DATA / "ashaar.train.tapt.diac.txt",
    "tapt_dev_diac":     DATA / "ashaar.dev.tapt.diac.txt",
    "tapt_test_diac":    DATA / "ashaar.test.tapt.diac.txt",
    "tapt_train_nodiac": DATA / "ashaar.train.tapt.nodiac.txt",
    "tapt_dev_nodiac":   DATA / "ashaar.dev.tapt.nodiac.txt",
    "tapt_test_nodiac":  DATA / "ashaar.test.tapt.nodiac.txt",
    "stats_md": DATA / "stats.md",
}

USE_MAP = {
  "ashaar.clean.parquet": "Source of truth: analysis, QA, re-build any serialization.",
  "ashaar.train.parquet": "Poet-wise TRAIN table (80%); counts should match TAPT train headers.",
  "ashaar.dev.parquet":   "Poet-wise DEV table (10%); quick eval & overfit check.",
  "ashaar.test.parquet":  "Poet-wise TEST table (10%); final offline eval (no training).",
  "ashaar.train.tapt.nodiac.txt": "TAPT warm-up (no diacritics) — faster & stabler.",
  "ashaar.train.tapt.diac.txt":   "TAPT continuation/mix — keeps phonetics for meter/rhyme.",
  "ashaar.dev.tapt.*.txt":        "Dev loss tracking during TAPT.",
  "ashaar.test.tapt.*.txt":       "Optional sanity eval after TAPT/E2E.",
  "stats.md": "Human-readable snapshot of data health; not used in training.",
}

for k, p in FILES.items():
    print(f"{k:>18}  →  {p}  {'✅' if p.exists() else '❌'}")

print("\nWhere/why files are used:\n")
for k, v in USE_MAP.items():
    print(f"- {k}: {v}")


Using data path: /root/Shaer/data
     clean_parquet  →  /root/Shaer/data/ashaar.clean.parquet  ✅
       split_train  →  /root/Shaer/data/ashaar.train.parquet  ✅
         split_dev  →  /root/Shaer/data/ashaar.dev.parquet  ✅
        split_test  →  /root/Shaer/data/ashaar.test.parquet  ✅
   tapt_train_diac  →  /root/Shaer/data/ashaar.train.tapt.diac.txt  ✅
     tapt_dev_diac  →  /root/Shaer/data/ashaar.dev.tapt.diac.txt  ✅
    tapt_test_diac  →  /root/Shaer/data/ashaar.test.tapt.diac.txt  ✅
 tapt_train_nodiac  →  /root/Shaer/data/ashaar.train.tapt.nodiac.txt  ✅
   tapt_dev_nodiac  →  /root/Shaer/data/ashaar.dev.tapt.nodiac.txt  ✅
  tapt_test_nodiac  →  /root/Shaer/data/ashaar.test.tapt.nodiac.txt  ✅
          stats_md  →  /root/Shaer/data/stats.md  ✅

Where/why files are used:

- ashaar.clean.parquet: Source of truth: analysis, QA, re-build any serialization.
- ashaar.train.parquet: Poet-wise TRAIN table (80%); counts should match TAPT train headers.
- ashaar.dev.parquet: Poet-wise DEV t

In [12]:
# %%
TAG_RE   = re.compile(r'^\[METER=.*\] \[ERA=.*\] \[THEME=.*\] \[RHYME=.*\] \[POET=.*\]$')
BAYT_RE  = re.compile(r'^<BAYT> ')
END_RE   = re.compile(r'^<END>$')
BSEP_RE  = re.compile(r'^<\|bsep\|>$')

def scan_tapt(path, sample_blocks=7, seed=1337, max_lines=None):
    rng = random.Random(seed)
    headers=end_=bsep=lines=0
    inblk=False; bayts=0
    # reservoir of blocks (as short text preview)
    bucket=[]; seen=0

    with open(path, "r", encoding="utf-8") as f:
        block=[]
        for ln_no, line in enumerate(f, 1):
            if max_lines and ln_no > max_lines:
                break
            line=line.rstrip("\n"); lines+=1
            block.append(line)

            if TAG_RE.match(line):
                if inblk:
                    print(f"ERR: new header before END @ line {ln_no}"); 
                inblk=True; bayts=0; headers+=1

            elif BAYT_RE.match(line):
                if not inblk: print(f"ERR: BAYT outside block @ line {ln_no}")
                bayts+=1

            elif END_RE.match(line):
                if not inblk: print(f"ERR: END outside block @ line {ln_no}")
                inblk=False; end_+=1

            elif BSEP_RE.match(line):
                bsep+=1
                # finalize block preview
                seen += 1
                if len(bucket) < sample_blocks:
                    bucket.append(block[:12])  # keep first ~12 lines as preview
                else:
                    j = rng.randrange(seen)
                    if j < sample_blocks:
                        bucket[j] = block[:12]
                block.clear()

            else:
                # Allow only the four patterns
                if not (TAG_RE.match(line) or BAYT_RE.match(line) or END_RE.match(line) or BSEP_RE.match(line)):
                    # show a small snippet only
                    print(f"ERR: unexpected content @ {ln_no}: {line[:80]!r}")

    ok = (headers == end_ == bsep)
    print(f"{path.name}: headers={headers}, END={end_}, bsep={bsep}, lines={lines}, OK={ok}")
    print("\n--- samples ---")
    for i, blk in enumerate(bucket, 1):
        print(f"\n[BLOCK {i}]")
        for ln in blk:
            print(ln)
    return dict(headers=headers, end=end_, bsep=bsep, ok=ok)

# run per file (fast)
tapts = {}
for key in ["tapt_train_nodiac","tapt_train_diac","tapt_dev_nodiac","tapt_dev_diac","tapt_test_nodiac","tapt_test_diac"]:
    if FILES[key].exists():
        tapts[key] = scan_tapt(FILES[key], sample_blocks=1)


ERR: unexpected content @ 1431821: ''
ashaar.train.tapt.nodiac.txt: headers=201049, END=201049, bsep=201049, lines=6606406, OK=True

--- samples ---

[BLOCK 1]
[METER=UNK] [ERA=UNK] [THEME=UNK] [RHYME=ك] [POET=شاهر ذيب]
<BAYT> المطار
<BAYT> لا منتم... بمنتهى البرود!!
<BAYT> لا تعرف كيف تغري الآخر بمحبتك..
<BAYT> في رحابك تضيق النفس،
<BAYT> وتتطلع للفضاء.
<BAYT> أجوبك وحيدا إلا من وهج كآبتي،
<BAYT> فلا شيء يشدني إليك
<BAYT> حتى بلاطك اللامع وموظفوك المتأنقون.
<BAYT> هذه الوجوه الحيادية
<BAYT> أمر على معالمها
<BAYT> مثل قرصان بلا خبرة.
ERR: unexpected content @ 1431823: ''
ashaar.train.tapt.diac.txt: headers=201049, END=201049, bsep=201049, lines=6606697, OK=True

--- samples ---

[BLOCK 1]
[METER=UNK] [ERA=UNK] [THEME=UNK] [RHYME=ك] [POET=شاهر ذيب]
<BAYT> المَطار
<BAYT> لا مُنتمٍ... بمُنتهى البُرود!!
<BAYT> لا تَعرفُ كيفَ تُغري الآخرَ بمحبَّتكَ..
<BAYT> في رِحابِكَ تَضيقُ النَّفسُ،
<BAYT> وتتطلَّعُ للفضاء.
<BAYT> أجوْبكَ وحيداً إلا مِن وهْجِ كآبتي،
<BAYT> فلا شيءَ يشدُّني إليكَ
<BAYT> ح

In [5]:
# %%
def count_headers(path):
    c = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if TAG_RE.match(line):
                c += 1
    return c

def compare_split(split_name, parquet_path, tapt_nodiac_path, tapt_diac_path):
    df = pd.read_parquet(parquet_path)
    n_rows = len(df)
    n_heads_nodiac = count_headers(tapt_nodiac_path) if tapt_nodiac_path.exists() else None
    n_heads_diac   = count_headers(tapt_diac_path)   if tapt_diac_path.exists() else None
    print(f"{split_name.upper():5} | parquet rows={n_rows:6d} | TAPT nodiac headers={n_heads_nodiac} | TAPT diac headers={n_heads_diac}")

compare_split("train", FILES["split_train"], FILES["tapt_train_nodiac"], FILES["tapt_train_diac"])
compare_split("dev",   FILES["split_dev"],   FILES["tapt_dev_nodiac"],   FILES["tapt_dev_diac"])
compare_split("test",  FILES["split_test"],  FILES["tapt_test_nodiac"],  FILES["tapt_test_diac"])


TRAIN | parquet rows=201049 | TAPT nodiac headers=201049 | TAPT diac headers=201049
DEV   | parquet rows= 27760 | TAPT nodiac headers=27760 | TAPT diac headers=27760
TEST  | parquet rows= 25800 | TAPT nodiac headers=25800 | TAPT diac headers=25800


In [6]:
# %%
# crude diac count function
DIAC_RE = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]')

def diac_ratio_of_block_lines(path, k=2000):
    # returns mean/median diacritics-per-char across k lines (sampled)
    total_d=0; total_c=0; n=0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.startswith("<BAYT> "):
                s = line[7:].rstrip("\n")
                total_d += len(DIAC_RE.findall(s))
                total_c += max(1, len(s))
                n += 1
                if n >= k:
                    break
    mean = (total_d/total_c) if total_c else 0.0
    return dict(lines=n, mean_diac_ratio=mean)

if FILES["tapt_train_diac"].exists():
    print("train.diac:", diac_ratio_of_block_lines(FILES["tapt_train_diac"]))
if FILES["tapt_train_nodiac"].exists():
    print("train.nodiac:", diac_ratio_of_block_lines(FILES["tapt_train_nodiac"]))


train.diac: {'lines': 2000, 'mean_diac_ratio': 0.22267047774601878}
train.nodiac: {'lines': 2000, 'mean_diac_ratio': 0.0}


In [8]:
# %%
# choose a tokenizer to estimate training cost
from transformers import AutoTokenizer
TOKENIZER_ID = "Navid-AI/Yehia-7B-preview"  # or "hammh0a/Hala-9B"
tok = AutoTokenizer.from_pretrained(TOKENIZER_ID, use_fast=True, trust_remote_code=True)

def estimate_tokens(path, n_blocks=1000):
    # stream first n_blocks and estimate tokens
    blocks = 0
    total_toks = 0
    buf = []
    with open(path, "r", encoding="utf-8") as f:
        for ln in f:
            buf.append(ln)
            if ln.strip() == "<|bsep|>":
                text = "".join(buf)
                total_toks += len(tok.encode(text))
                blocks += 1
                buf.clear()
                if blocks >= n_blocks:
                    break
    avg = total_toks / max(1, blocks)
    print(f"{path.name}: blocks={blocks}, est avg tokens/block ≈ {avg:.1f}, total tokens≈{total_toks:,}")
    return avg, total_toks

if FILES["tapt_train_nodiac"].exists():
    estimate_tokens(FILES["tapt_train_nodiac"], n_blocks=400)
if FILES["tapt_train_diac"].exists():
    estimate_tokens(FILES["tapt_train_diac"], n_blocks=400)


ashaar.train.tapt.nodiac.txt: blocks=400, est avg tokens/block ≈ 226.5, total tokens≈90,584
ashaar.train.tapt.diac.txt: blocks=400, est avg tokens/block ≈ 296.9, total tokens≈118,780


In [10]:
# %%
plan = {
  "TAPT warm-up (optional but recommended)": [
    "ashaar.train.tapt.nodiac.txt",
    "ashaar.dev.tapt.nodiac.txt"
  ],
  "TAPT continue / mix (phonetic)": [
    "ashaar.train.tapt.diac.txt",
    "ashaar.dev.tapt.diac.txt"
  ],
  "Final offline eval (no training)": [
    "ashaar.test.tapt.nodiac.txt",
    "ashaar.test.tapt.diac.txt"
  ],
  "Analysis / future SFT construction": [
    "ashaar.clean.parquet",
    "ashaar.train.parquet",
    "ashaar.dev.parquet",
    "ashaar.test.parquet"
  ],
  "Documentation": [
    "stats.md"
  ],
}
print(json.dumps(plan, indent=2, ensure_ascii=False))


{
  "TAPT warm-up (optional but recommended)": [
    "ashaar.train.tapt.nodiac.txt",
    "ashaar.dev.tapt.nodiac.txt"
  ],
  "TAPT continue / mix (phonetic)": [
    "ashaar.train.tapt.diac.txt",
    "ashaar.dev.tapt.diac.txt"
  ],
  "Final offline eval (no training)": [
    "ashaar.test.tapt.nodiac.txt",
    "ashaar.test.tapt.diac.txt"
  ],
  "Analysis / future SFT construction": [
    "ashaar.clean.parquet",
    "ashaar.train.parquet",
    "ashaar.dev.parquet",
    "ashaar.test.parquet"
  ],
  "Documentation": [
    "stats.md"
  ]
}


In [16]:
from datasets import load_dataset

ds = load_dataset("arbml/ashaar", split="train")

def find_poem_example(poet_substr="شاهر ذيب", first_line_substr="المطار", max_rows=300000):
    for i, row in enumerate(ds):
        if i >= max_rows:
            break
        poet = row.get("poet name", "") or ""
        verses = row.get("poem verses", []) or []
        if poet_substr in poet and verses and first_line_substr in verses[0]:
            print("✅ Found at index:", i)
            return row
    return None

row = find_poem_example()

print("FOUND ROW? ", row is not None)
if row:
    print("---- META FROM HF DATASET ----")
    print("title:", row["poem title"])
    print("meter:", row["poem meter"])
    print("era:", row["poet era"])
    print("theme:", row["poem theme"])
    print("poet:", row["poet name"])
    print("\n---- VERSES FROM HF DATASET ----")
    for v in row["poem verses"]:
        print(v)


FOUND ROW?  False


In [ ]:
from pathlib import Path

tapt_path = Path("data/ashaar.train.tapt.nodiac.txt")

def extract_block_for_poem(path, poet_substr="شاهر ذيب", first_line_substr="المطار"):
    block = []
    found = False

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")

            # block separator → reset
            if line.strip() == "<|bsep|>":
                if block:
                    # inspect finished block
                    header = block[0] if block else ""
                    if poet_substr in header:
                        for ln in block[1:]:
                            if ln.startswith("<BAYT>") and first_line_substr in ln:
                                found = True
                                return block
                    block = []
            else:
                block.append(line)

    if block and not found:
        header = block[0]
        if poet_substr in header:
            for ln in block[1:]:
                if ln.startswith("<BAYT>") and first_line_substr in ln:
                    return block

    return None

block = extract_block_for_poem(ROOT.parent/tapt_path)

print("FOUND BLOCK? ", block is not None)
if block:
    print("---- RAW TAPT BLOCK ----")
    for ln in block:
        print(ln)


FileNotFoundError: [Errno 2] No such file or directory: 'data/ashaar.train.tapt.nodiac.txt'